# Chapter 45: Semantic & Instance Segmentation
## Multimedia & Computer Vision Learning Hub — ITERA
**Author:** Aji Pamungkas Tri Nurcahyo
**Objective:** Memahami segmentasi semantik dan instansi pada tingkat piksel, merumuskan arsitektur encoder-decoder U-Net dengan skip connections, mengoptimalkan fungsi Soft Dice Loss, serta merancang pipeline segmentasi deep learning riil (Torchvision DeepLabV3 / U-Net).

### 1. Landasan Teori & Formulasi Soft Dice Loss
Dalam segmentasi piksel, koefisien Sørensen–Dice kontinu diformulasikan sebagai:
$$\mathcal{L}_{\text{Dice}} = 1 - \frac{2 \sum_{i=1}^N p_i g_i + \epsilon}{\sum_{i=1}^N p_i^2 + \sum_{i=1}^N g_i^2 + \epsilon}$$
di mana $p_i \in [0, 1]$ adalah prediksi probabilitas model dan $g_i \in \{0, 1\}$ adalah label biner ground truth.

Metrik Mean Intersection over Union (mIoU) tingkat piksel di seluruh $C$ kelas:
$$\text{mIoU} = \frac{1}{C} \sum_{c=1}^C \frac{TP_c}{TP_c + FP_c + FN_c}$$

In [ ]:
# 2. Setup Lingkungan & Dual-Engine Check
import numpy as np
import sys

print(f"Python Version: {sys.version}")
print(f"NumPy Version: {np.__version__}")

HAS_TORCH = False
try:
    import torch
    import torch.nn as nn
    import torchvision.models.segmentation as seg_models
    HAS_TORCH = True
    print("Torchvision Segmentation Models: INSTALLED & AVAILABLE")
except ImportError:
    print("INFO: PyTorch/torchvision belum terpasang di host lokal.")
    print("Notebook beroperasi dalam Mode Analisis Algoritmik Deterministik (U-Net Skip + Soft Dice + mIoU).")

In [ ]:
# 3. Simulasi Konkatenasi Skip-Connection U-Net
def unet_skip_fusion(enc_features, dec_features):
    return np.concatenate([dec_features, enc_features], axis=1)

enc_feat = np.ones((1, 64, 64, 64), dtype=np.float32)
dec_feat = np.ones((1, 64, 64, 64), dtype=np.float32) * 2.0
fused = unet_skip_fusion(enc_feat, dec_feat)
print("Bentuk Tensor Skip-Fusion U-Net:", fused.shape)
print(f"Jumlah kanal hasil fusi: {fused.shape[1]} (Kombinasi Decoder 64 + Encoder 64)")
assert fused.shape == (1, 128, 64, 64)

In [ ]:
# 4. Implementasi & Perhitungan Soft Dice Loss
def soft_dice_loss(pred_probs, ground_truth, eps=1.0):
    p = pred_probs.flatten()
    g = ground_truth.flatten()
    intersection = np.sum(p * g)
    cardinality = np.sum(p**2) + np.sum(g**2)
    dice = (2.0 * intersection + eps) / (cardinality + eps)
    return 1.0 - dice

gt = np.array([[1, 1], [0, 0]], dtype=np.float64)
probs = np.array([[0.90, 0.80], [0.20, 0.10]], dtype=np.float64)

loss = soft_dice_loss(probs, gt, eps=1.0)
print(f"Prediksi Probabilitas:\n{probs}")
print(f"Ground Truth:\n{gt}")
print(f"Nilai Soft Dice Loss: {loss:.4f} (Ekspektasi: 0.0222)")
assert np.isclose(loss, 0.02222, atol=1e-4)

In [ ]:
# 5. Evaluasi Metrik Mean IoU (mIoU) Tingkat Piksel
def compute_miou(pred_mask, gt_mask, num_classes=2):
    ious = []
    for c in range(num_classes):
        tp = np.sum((pred_mask == c) & (gt_mask == c))
        fp = np.sum((pred_mask == c) & (gt_mask != c))
        fn = np.sum((pred_mask != c) & (gt_mask == c))
        union = tp + fp + fn
        iou = tp / union if union > 0 else 1.0
        ious.append(iou)
    return ious, np.mean(ious)

pred_bin = (probs >= 0.50).astype(int)
class_ious, miou = compute_miou(pred_bin, gt.astype(int), num_classes=2)
print(f"IoU Kelas Latar Belakang (Class 0): {class_ious[0]:.4f}")
print(f"IoU Kelas Objek Depan (Class 1):    {class_ious[1]:.4f}")
print(f"Mean IoU (mIoU) Keseluruhan:       {miou:.4f}")
assert np.isclose(miou, 1.0)

In [ ]:
# 6. Real PyTorch DeepLabV3-ResNet50 / U-Net Segmentation Pipeline
if HAS_TORCH:
    # Load pretrained DeepLabV3 with MobileNetV3-Large backbone (Lightweight ~11M params)
    deeplab_model = seg_models.deeplabv3_mobilenet_v3_large(weights=seg_models.DeepLabV3_MobileNet_V3_Large_Weights.DEFAULT)
    deeplab_model.eval()
    print("Pretrained DeepLabV3-MobileNetV3 Loaded Successfully for 21 PASCAL VOC Classes.")
    
    # Sintesis citra input RGB 1x3x256x256
    x_dummy = torch.randn(1, 3, 256, 256)
    with torch.no_grad():
        output = deeplab_model(x_dummy)['out'] # Output shape: (1, 21, 256, 256)
        pred_mask = torch.argmax(output.squeeze(), dim=0) # Mask shape: (256, 256)
    print(f"Output Logits Shape: {output.shape}")
    print(f"Predicted Pixel Mask Shape: {pred_mask.shape} (256x256 piksel)")
else:
    print("Panduan Arsitektur Segmentasi Riil (DeepLabV3 / U-Net):")
    print("1. Model Pretrained: torchvision.models.segmentation.deeplabv3_mobilenet_v3_large(weights=DEFAULT)")
    print("2. Input Format: Tensor NCHW (1, 3, H, W) dinormalisasi mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225].")
    print("3. Output Tensor: Logit per kelas (1, num_classes, H, W) lalu diambil 'argmax(dim=1)'.")
    print("4. Penanganan Batas: Terapkan 'ignore_index = 255' pada Loss Cross-Entropy untuk garis tepi PASCAL VOC.")
    print("5. Jalankan di Colab/GPU untuk visualisasi segmentasi objek multi-kelas secara interaktif.")

### 7. Analisis Hasil, Evaluasi & Rekomendasi
1. **Keunggulan Skip Connection**: Meniadakan kehilangan informasi spasial frekuensi tinggi selama kompresi representasi pada lapisan bottleneck.
2. **Ketahanan Soft Dice Loss**: Tetap stabil dan konvergen bahkan ketika piksel objek target hanya menempati sebagian kecil (<1%) dari keseluruhan kanvas gambar.
3. **Evaluasi mIoU**: Standar baku evaluasi yang memberikan penilaian seimbang antara ketepatan area (*precision*) dan kelengkapan cakupan masker (*recall*).